In [1]:
import pandas as pd
from rapidfuzz import fuzz, process
import python_files.functions_barcelona as fb

import time
import folium
from geopy.geocoders import Nominatim
from geopy.exc import GeocoderTimedOut, GeocoderRateLimited

In [2]:
##Functions

def safe_reverse_geocode(lat, lon):
    try:
        # Increase timeout to 10 seconds instead of 1
        location = geolocator.reverse((lat, lon), timeout=10)
        
        return location
    except GeocoderTimedOut:
        print("Server was too slow, trying again...")
        time.sleep(2)
        return safe_reverse_geocode(lat, lon)


In [16]:
year = 2025
geolocator = Nominatim(user_agent="BarcelonaAccidents Analysis")
accidents = pd.read_csv(fb.FILES_PATH / 'accidents_weather2025.csv')

In [1081]:
# 1. Load the Open Data BCN master registry and accidents

bcn_streets = pd.read_csv(fb.FILES_PATH /'carrerer.csv')
accidents = pd.read_csv(fb.FILES_PATH / f'accidents_weather2025.csv')
# Requires columns: codi_via, nom_oficial, nom_curt

# Lowercase and clean names for robust dictionary lookup
bcn_streets['clean_official'] = bcn_streets['nom_oficial'].str.lower().str.strip()
bcn_streets['clean_short'] = bcn_streets['nom_curt'].str.lower().str.strip()
official_lookup = dict(zip(bcn_streets['clean_official'], zip(bcn_streets['codi_via'], bcn_streets['nom_oficial'])))
short_lookup = dict(zip(bcn_streets['clean_short'], zip(bcn_streets['codi_via'], bcn_streets['nom_oficial'])))

# 2. Extract a list of all clean official names for Fuzzy Matching fallback
all_official_names = list(official_lookup.keys())
all_short_names = list(short_lookup.keys())

my_df =accidents[['street_name']].copy()
my_df.columns = ['raw_location']

# Helper function to find a code via exact lookup or rapidfuzz fallback
def get_street_details_hierarchical(street_name, confidence_cutoff=75.0):
    if not street_name:
        return None, None, 0
    
    st_clean = street_name.lower().strip()
    
    # ---- TIER 1: Exact Match on Short Names ----
    if st_clean in short_lookup:
        code, official_name = short_lookup[st_clean]
        return int(code), official_name, 100.0

    # ---- TIER 2: Exact Match on Official Names ----
    if st_clean in official_lookup:
        code, official_name = official_lookup[st_clean]
        return int(code), official_name, 100.0

    # ---- TIER 3: Fuzzy Match on Short Names (Handles short typos) ----
    match_short = process.extractOne(st_clean, all_short_names, scorer=fuzz.token_sort_ratio, score_cutoff=confidence_cutoff)
    if match_short:
        matched_key = match_short[0]
        score = match_short[1]
        code, official_name = short_lookup[matched_key]
        return int(code), official_name, float(score)

    # ---- TIER 4: Fuzzy Match on Official Names (Handles long typos) ----
    match_official = process.extractOne(st_clean, all_official_names, scorer=fuzz.token_sort_ratio, score_cutoff=confidence_cutoff)
    if match_official:
        matched_key = match_official[0]
        score =match_official[1]
        code, official_name = official_lookup[matched_key]
        
        return int(code), official_name, float(score)
        
    return None, None, 0.0
    
def parse_and_assign_codes(address_str):
    if pd.isna(address_str):
        return pd.Series([None, None, None, None, "Single Street", "Empty Row", 0.0])
    
    cleaned = str(address_str).lower()
    for prefix in ["carrer de ", "carrer del ", "avinguda de ", "av. ", "passeig de ", "pg. "]:
        cleaned = cleaned.replace(prefix, "")

    delimiters = ['/', ' amb ', ' con ', '&']
    split_delimiter = None
    for d in delimiters:
        if d in cleaned:
            split_delimiter = d
            break
    # Crossroads Parsing
    if split_delimiter:
        parts = cleaned.split(split_delimiter)
        st1, st2 = parts[0].strip(), parts[1].strip()
        
        code1, name1, score1 = get_street_details_hierarchical(st1)
        code2, name2, score2 = get_street_details_hierarchical(st2)

        if code1 and code2:
            avg_score = (score1 + score2) / 2
            return pd.Series([code1, code2, name1, name2, "Crossroad", "Successfully Matched", avg_score])
        else:
            return pd.Series([code1, code2, name1, name2, "Crossroad", "Error: Unmatched segments", min(score1, score2)])

    else:
        code1, name1, score1 = get_street_details_hierarchical(cleaned.strip())
        if code1:
            return pd.Series([code1, None, name1, None, "Single Street", f"Matched ({name1})", score1])
        else:
            return pd.Series([None, None, None, None, "Single Street", "Error: Unmatched", 0.0])
        
processed_data = my_df['raw_location'].apply(parse_and_assign_codes)

new_cols = ['code_via_1', 'code_via_2', 'official_name_1', 
            'official_name_2', 'location_type', 'match_status', 
            'confidence_score']
processed_data.columns = new_cols
final_df = pd.concat([my_df['raw_location'], processed_data],axis =1)


# Formatting output display view
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

assert final_df.confidence_score.mean() >90, "Confidence Score too low"
if final_df[final_df.match_status == 'Error: Unmatched'].shape[0] == 0:
    final_df.drop_duplicates().to_csv(fb.FILES_PATH / f'Cleaned_streets_{year}.csv',index=False)
else:
    print(f'Check this issue {final_df[final_df.match_status == "Error: Unmatched"].shape[0]}')

93.34240660385737


,raw_location,code_via_1,code_via_2,official_name_1,official_name_2,location_type,match_status,confidence_score
0,Meridiana,209900.0,NaN,Avinguda Meridiana,None,Single Street,Matched (Avinguda Meridiana),85.714286
1,Trinxant,346402.0,NaN,Carrer de Trinxant,None,Single Street,Matched (Carrer de Trinxant),100.000000
2,Sant Antoni Maria Claret,297001.0,NaN,Carrer de Sant Antoni Maria Claret,None,Single Street,Matched (Carrer de Sant Antoni Maria Claret),85.714286
3,Freser,134801.0,NaN,Carrer del Freser,None,Single Street,Matched (Carrer del Freser),100.000000
4,Indústria,161407.0,NaN,Carrer de la Indústria,None,Single Street,Matched (Carrer de la Indústria),100.000000
5,Dos de Maig,105304.0,NaN,Carrer del Dos de Maig,None,Single Street,Matched (Carrer del Dos de Maig),100.000000
6,Aragó,18505.0,NaN,Carrer d'Aragó,None,Single Street,Matched (Carrer d'Aragó),100.000000
7,Meridiana,209900.0,NaN,Avinguda Meridiana,None,Single Street,Matched (Avinguda Meridiana),85.714286
8,Eterna Memòria,117802.0,NaN,Carrer de l'Eterna Memòria,None,Single Street,Matched (Carrer de l'Eterna Memòria),100.000000
9,Espronceda,115603.0,NaN,Carrer d'Espronceda,None,Single Street,Matched (Carrer d'Espronceda),100.000000


In [7]:
##Fixing desconegut District
unique_districts = accidents.district.unique().tolist()
try:
    unique_districts.remove('Desconegut')
except:
    print("DONE FIXING DISTRICTS")
unique_districts = [dis.lower() for dis in unique_districts]
unknown_districts = accidents[accidents.district == 'Desconegut']
print(unknown_districts.shape)
to_check = []

for idx, row in unknown_districts.head(20).iterrows():
    print(idx, row['lat'],row['lon'])
    result = safe_reverse_geocode(row['lat'], row['lon'])
    # print(row['street_name'].upper(), result.raw['address'])
    if 'suburb'  in result.raw['address']:
        print("SUBURB: ",result.raw['address']['suburb'])
        accidents.loc[idx, 'district'] = result.raw['address']['suburb'].strip()
    elif 'town' in  result.raw['address']:
        print("TOWN: ",result.raw['address']['town'])
        accidents.loc[idx, 'district'] = result.raw['address']['town'].strip()
    elif 'city' in result.raw['address']:
        print("CITY: ",result.raw['address']['city'])
        accidents.loc[idx, 'district'] = result.raw['address']['city'].strip()
    else:
        print("ALARMMMMMMMMMMMMMM",idx)
        to_check.append((idx,result.raw['address']))
    time.sleep(1.5)
len(to_check)

DONE FIXING DISTRICTS
(0, 29)


0

In [67]:
len(sorted(unique_neigh)), sorted(unique_neigh)

(76,
 ['baró de viver',
  'can baró',
  'can peguera',
  'canyelles',
  'ciutat meridiana',
  'diagonal mar i el front marítim del poblenou',
  'el baix guinardó',
  'el barri gòtic',
  'el besòs i el maresme',
  'el bon pastor',
  "el camp d'en grassot i gràcia nova",
  "el camp de l'arpa del clot",
  'el carmel',
  'el clot',
  'el coll',
  'el congrés i els indians',
  'el fort pienc',
  'el guinardó',
  'el parc i la llacuna del poblenou',
  'el poble sec',
  'el poble-sec',
  'el poblenou',
  'el putxet i el farró',
  'el raval',
  'el turó de la peira',
  'horta',
  'hostafrancs',
  "l'antiga esquerra de l'eixample",
  'la barceloneta',
  'la bordeta',
  'la clota',
  "la dreta de l'eixample",
  "la font d'en fargues",
  'la font de la guatlla',
  'la guineueta',
  'la marina de port',
  'la marina del prat vermell',
  'la maternitat i sant ramon',
  "la nova esquerra de l'eixample",
  'la prosperitat',
  'la sagrada família',
  'la sagrera',
  'la salut',
  'la teixonera',
  'la

In [92]:
##Fixing Neighborhood desconegut
unique_neigh = accidents.neighborhood.unique().tolist()
try:
    unique_neigh.remove('Desconegut')
except:
    print("DONE FIXING NEIGHBORHOODS")
unique_neigh = [dis.lower() for dis in unique_neigh]
unknown_hoods = accidents[accidents.neighborhood == 'Desconegut']
print(unknown_hoods.shape)
to_check = []

for idx, row in unknown_hoods.head(10).iterrows():
    print(idx, row['lat'],row['lon'])
    result = safe_reverse_geocode(row['lat'], row['lon'])
    # print(row['street_name'].upper(), result.raw['address'])
    if 'quarter' in result.raw['address']:
        print("QUARTER: ",result.raw['address']['quarter'])
        accidents.loc[idx, 'neighborhood'] = result.raw['address']['quarter'].strip()
    elif 'town' in  result.raw['address']:
        print("TOWN: ",result.raw['address']['town'])
        accidents.loc[idx, 'neighborhood'] = result.raw['address']['town'].strip()
    elif 'city' in result.raw['address']:
        print("CITY: ",result.raw['address']['city'])
        accidents.loc[idx, 'neighborhood'] = result.raw['address']['city'].strip()
    else:
        print("ALARMMMMMMMMMMMMMM",idx)
        to_check.append((idx,result.raw['address']))
    time.sleep(1.5)
len(to_check)


DONE FIXING DISTRICTS
(0, 29)


0

In [91]:
accidents.to_csv(fb.FILES_PATH / 'accidents_weather2025.csv',index=False)
accidents = pd.read_csv(fb.FILES_PATH / 'accidents_weather2025.csv')

In [995]:
#Example usage in a loop
coordinates = [coor]
for lat, lon in coordinates:
    result = safe_reverse_geocode(lat, lon)
    print(result,'+++++++++','\n', result[0].split(',')[1])   
    
    # CRITICAL: Sleep for 1+ seconds to respect the usage policy
    time.sleep(1.5)

for corr in pending_acc.iterrows():
    print(corr)
    


In [1112]:
##Printing data points
df = accidents.loc[to_check]

# 2. Create the base map centered on Barcelona (Lat: 41.3851, Lon: 2.1734)
barcelona_map = folium.Map(
    location=[41.3851, 2.1734], 
    zoom_start=13, 
    tiles='OpenStreetMap'
)

# 3. Add markers to the map
for index, row in df.iterrows():
    folium.Marker(
        location=[row['lat'], row['lon']],
        popup=(row['street_name'],row['lon']), # Text displayed when the marker is clicked
        tooltip='Click to see name', # Text displayed when hovering
        icon=folium.Icon(color='red', icon='info-sign')
    ).add_to(barcelona_map)

# 4. Save and display the map
barcelona_map.save('barcelona_map.html')
# To display in a Jupyter Notebook, simply type: barcelona_map
barcelona_map